In [282]:
import HEDAC_basic
import environment_modelling

import matplotlib.pyplot as plt
import matplotlib as mpl

import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
import contextily as ctx

import numpy as np 
np.set_printoptions(precision=4)
mpl.rcParams['axes.linewidth'] = 3
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['axes.titlepad'] = 8.0
mpl.rcParams['xtick.major.size'] = 6
mpl.rcParams['xtick.major.width'] = 3
mpl.rcParams['xtick.labelsize'] = 20
mpl.rcParams['ytick.major.size'] = 6
mpl.rcParams['ytick.major.width'] = 3
mpl.rcParams['ytick.labelsize'] = 20
mpl.rcParams['lines.markersize'] = 5
mpl.rcParams['legend.fontsize'] = 15

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.sans-serif": "Palantino",
})



In [283]:
# Discretize the search space into 100-by-100 mesh grids
polygon_file = "data/DemaScenarios/FlatTerrainNature.geojson"
env = environment_modelling.Environment(polygon_file)


2024-10-09 12:57:54,731 - INFO - Query region bounds: (9.416186034424953, 54.952615782731385, 9.42699155445743, 54.96100438506525) (runpy.py:41)
/home/kang/.local/lib/python3.10/site-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/kang/.local/lib/python3.10/site-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/kang/.local/lib/python3.10/site-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/kang/.local/lib/python3.10/site-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. U

In [284]:

filter_width_meters = 50  # Example width in meters
filter_width_pixels = filter_width_meters / env.meter_per_bin

# Calculate the sigma value for the Gaussian filter
sigma = filter_width_pixels / (2 * np.sqrt(2 * np.log(2)))

heatmap = gaussian_filter(env.heatmaps["roads"],sigma=sigma) + gaussian_filter(env.heatmaps["wetland"],sigma=sigma) 
heatmap = np.array(heatmap, dtype=np.float64)
heatmap /= np.sum(heatmap)
# # Normalize the heatmap
# heatmap -= np.min(heatmap)
# heatmap /= np.max(heatmap)

In [ ]:
### We are going to use 10 coefficients per dimension --- so 100 index vectors in total
num_k_per_dim = 20
ks_dim1, ks_dim2 = np.meshgrid(
    np.arange(num_k_per_dim), np.arange(num_k_per_dim)
)
ks = np.array([ks_dim1.ravel(), ks_dim2.ravel()]).T  # this is the set of all index vectors

# Use the heatmap instead of the grids
pdf_vals = heatmap.ravel()  # Flatten the heatmap to match the shape of grids
pdf_gt = heatmap # ground truth density function

# Define the dimensions based on the heatmap size
L_list = np.array([heatmap.shape[1], heatmap.shape[0]])  # boundaries for each dimension

# Discretize the search space into mesh grids based on the heatmap size
grids_x, grids_y = np.meshgrid(
    np.linspace(0, L_list[0], heatmap.shape[1]),
    np.linspace(0, L_list[1], heatmap.shape[0])
)

grids = np.array([grids_x.ravel(), grids_y.ravel()]).T
dx = 1.0 / (heatmap.shape[1] - 1)
dy = 1.0 / (heatmap.shape[0] - 1)  # the resolution of the grids

# Compute the coefficients
coefficients = np.zeros(ks.shape[0])  # number of coefficients matches the number of index vectors
for i, k_vec in enumerate(ks):
    # step 1: evaluate the fourier basis function over all the grid cells
    fk_vals = np.prod(np.cos(np.pi * k_vec / L_list * grids), axis=1)  # we use NumPy's broadcasting feature to simplify computation
    hk = np.sqrt(np.sum(np.square(fk_vals)) * dx * dy)  # normalization term
    fk_vals /= hk

    # step 3: approximate the integral through the Riemann sum for the coefficient
    phik = np.sum(fk_vals * pdf_vals) * dx * dy 
    coefficients[i] = phik

print('First 5 coefficients: ', coefficients[:5])

First 5 coefficients:  [ 1.3994e-05 -3.1175e-06 -6.5760e-06  4.0476e-06 -5.0554e-06]


In [ ]:
### We can verify the correctness of the coefficients by reconstructing the probability
### density function through the coefficients

pdf_recon = np.zeros(grids.shape[0])
for i, (phik, k_vec) in enumerate(zip(coefficients, ks)):
    fk_vals = np.prod(np.cos(np.pi * k_vec / L_list * grids), axis=1)
    hk = np.sqrt(np.sum(np.square(fk_vals)) * dx * dy)
    fk_vals /= hk
    
    pdf_recon += phik * fk_vals 

# visualize for comparison
fig, axes = plt.subplots(1, 2, figsize=(9,5), dpi=70, tight_layout=True)

ax = axes[0]
ax.set_aspect('equal')
ax.set_xlim(0.0, L_list[1])
ax.set_ylim(0.0, L_list[0])
ax.set_title('Original PDF')
ax.contourf(grids_y, grids_x, pdf_gt.reshape(grids_y.shape), cmap='Reds')

ax = axes[1]
ax.set_aspect('equal')
ax.set_xlim(0.0, L_list[1])
ax.set_ylim(0.0, L_list[0])
ax.set_title('Reconstructed PDF')
ax.contourf(grids_y, grids_x, pdf_recon.reshape(grids_y.shape), cmap='Blues')

plt.show()
plt.close()

In [275]:

def calculate_ergodic_metric(target_distribution, agent_distribution):
    # normalize both distributions
    # target_distribution -= np.min(target_distribution)
    # target_distribution /= np.max(target_distribution)
    # agent_distribution -= np.min(agent_distribution)
    # agent_distribution /= np.max(agent_distribution)
    
    num_k_per_dim = 20
    ks_dim1, ks_dim2 = np.meshgrid(
        np.arange(num_k_per_dim), np.arange(num_k_per_dim)
    )
    ks = np.array([ks_dim1.ravel(), ks_dim2.ravel()]).T
    
    # Discretize the search space into mesh grids based on the heatmap size
    grids_x, grids_y = np.meshgrid(
        np.linspace(0, L_list[0], heatmap.shape[1]),
        np.linspace(0, L_list[1], heatmap.shape[0])
    )
    grids = np.array([grids_x.ravel(), grids_y.ravel()]).T
    
    # Assert that the target and agent distributions are the same size
    assert target_distribution.shape == agent_distribution.shape
    target_coefficients = get_fourier_coef(target_distribution, grids, num_k_per_dim)
    trajectory_coef = get_fourier_coef(agent_distribution, grids, num_k_per_dim)
    
    lambda_k = np.power(1.0 + np.linalg.norm(ks, axis=1), -3/2.0)
    erg_metric = np.sum(lambda_k * np.square(target_coefficients - trajectory_coef))
    return erg_metric

def get_fourier_coef(heatmap, grids, num_k_per_dim=30):
    ks_dim1, ks_dim2 = np.meshgrid(
        np.arange(num_k_per_dim), np.arange(num_k_per_dim)
    )
    ks = np.array([ks_dim1.ravel(), ks_dim2.ravel()]).T  # this is the set of all index vectors
    # Use the heatmap instead of the grids
    pdf_vals = heatmap.ravel() 
    # Define the dimensions based on the heatmap size
    L_list = np.array([heatmap.shape[1], heatmap.shape[0]])  # boundaries for each dimension
    dx = 1.0 / (heatmap.shape[1] - 1)
    dy = 1.0 / (heatmap.shape[0] - 1)  # the resolution of the grids

    # Compute the coefficients
    coefficients = np.zeros(ks.shape[0])  # number of coefficients matches the number of index vectors
    for i, k_vec in enumerate(ks):
        # step 1: evaluate the fourier basis function over all the grid cells
        fk_vals = np.prod(np.cos(np.pi * k_vec / L_list * grids), axis=1)  # we use NumPy's broadcasting feature to simplify computation
        hk = np.sqrt(np.sum(np.square(fk_vals)) * dx * dy)  # normalization term
        fk_vals /= hk
        
        # step 3: approximate the integral through the Riemann sum for the coefficient
        phik = np.sum(fk_vals * pdf_vals) * dx * dy 
        coefficients[i] = phik
    return coefficients



In [ ]:
# simulate the trajectory 
tsteps = 500
dt = 1/tsteps
s_traj = []
for t in range(tsteps):
    # Sample points based on the probabilities in pdf_gt
    probabilities = pdf_gt.ravel()
    probabilities /= np.sum(probabilities)  # Ensure the probabilities sum to 1
    index = np.random.choice(len(probabilities), p=probabilities)
    y_idx, x_idx = np.unravel_index(index, pdf_gt.shape)
    st_new = np.array([x_idx, y_idx])
    s_traj.append(st_new)
s_traj = np.array(s_traj)

# compute the coefficient of the trajectory
traj_coefficients = np.zeros(ks.shape[0])
for i, k_vec in enumerate(ks):
    # step 1: evaluate the fourier basis function over all the grid cells
    fk_vals = np.prod(np.cos(np.pi * k_vec / L_list * s_traj), axis=1) # we use NumPy's broadcasting feature to simplify computation
    hk = np.sqrt(np.sum(np.square(fk_vals)) * dt)  # normalization term
    fk_vals /= hk
    # step 3: approximate the integral through the Riemann sum for the coefficient
    phik = np.sum(fk_vals) * dt 
    traj_coefficients[i] = phik

phi_recon = np.zeros(grids.shape[0])
for i, (ck, k_vec) in enumerate(zip(traj_coefficients, ks)):
    fk_vals = np.prod(np.cos(np.pi * k_vec / L_list * grids), axis=1)
    hk = np.sqrt(np.sum(np.square(fk_vals)) * dx * dy)
    fk_vals /= hk
    phi_recon += ck * fk_vals 

# Visualize the reconstructed phi function
# visualize for comparison
fig, axes = plt.subplots(1, 3, figsize=(18,10), dpi=70, tight_layout=True)

# Normalize pdf_gt
pdf_gt -= np.min(pdf_gt)
pdf_gt /= np.max(pdf_gt)

# Normalize pdf_recon
pdf_recon -= np.min(pdf_recon)
pdf_recon /= np.max(pdf_recon)

# Normalize phi_recon
phi_recon -= np.min(phi_recon)
phi_recon /= np.max(phi_recon)

# plotting
ax = axes[0]
ax.set_aspect('equal')
ax.set_xlim(0.0, L_list[1])
ax.set_ylim(0.0, L_list[0])
ax.set_title('Ground truth PDF')
cbar = plt.colorbar(ax.contourf(grids_y, grids_x, pdf_gt, cmap='Reds'), ax=ax, shrink=0.5)
ax.contourf(grids_y, grids_x, pdf_gt, cmap='Reds')

ax = axes[1]
ax.set_aspect('equal')
ax.set_xlim(0.0, L_list[1])
ax.set_ylim(0.0, L_list[0])
ax.set_title('Reconstructed PDF')
cbar = plt.colorbar(ax.contourf(grids_y, grids_x, pdf_recon.reshape(grids_x.shape), cmap='Blues'), ax=ax, shrink=0.5)
ax.contourf(grids_y, grids_x, pdf_recon.reshape(grids_x.shape), cmap='Blues')

ax = axes[2]
ax.set_aspect('equal')
ax.set_xlim(0.0, L_list[1])
ax.set_ylim(0.0, L_list[0])
ax.set_title('Sampled PDF')
cbar = plt.colorbar(ax.contourf(grids_y, grids_x, phi_recon.reshape(grids_y.shape), cmap='Blues'), ax=ax, shrink=0.5)
cbar.set_label('Probability Density')

ax.plot(s_traj[:,1],s_traj[:,0], linestyle='', marker='o', color='k', alpha=0.1, label='Trajectory')
ax.contourf(grids_y, grids_x, phi_recon.reshape(grids_y.shape), cmap='Blues')

# Plot the value of the ergodic metric for the two last plots
# Make a copy of the distributions and parse them
pdf_gt_copy = pdf_gt.copy()
pdf_recon_copy = pdf_recon.copy().reshape(grids_x.shape)

ergodic_metric_recon = calculate_ergodic_metric(pdf_gt_copy, pdf_recon_copy)
ergodic_metric_sampled = calculate_ergodic_metric(pdf_gt, phi_recon.reshape(grids_x.shape))
# Write the ergodic metric underneath the plots
for ax, metric in zip(axes[1:], [ergodic_metric_recon, ergodic_metric_sampled]):
    ax.text(0.5, -0.1, f'Ergodic Metric: {metric:.4f}', transform=ax.transAxes, ha='center', fontsize=18)

plt.show()
plt.close()

In [269]:
# Convert the ergodicity into a reward


# Generate a trajectory using the HEDAC algorithm


# Generate a distribution of each trajectory and evaluate the ergodic metric

